# Variable pricing: one price, or a price for each day?

A theme park has a fixed daily capacity $C = 1000$ and faces different demand on each of seven days. Demand on day $i$ is linear in the price, $x_i \le D_i + m_i\, p$ with $m_i < 0$, and is also capped by capacity. The question is how to price across the week to maximize total revenue.

We solve three increasingly flexible versions as a nonlinear program (Pyomo + `ipopt`):

1. **A single price** charged every day.
2. **A different price each day** — variable pricing. This segments customers by *self-selection* (when they choose to visit), filling capacity on slow days and charging more on busy ones.
3. **Day-specific prices with diversion** — demand on day $i$ now also responds to the price *gap* to other days, so cutting one day's price pulls flexible customers away from the others.

This is the theme-park example from Phillips, Section 7.5. As there, moving from one price to seven raises revenue substantially even though the *average* price paid barely changes — the gain comes from better matching price to each day's willingness to pay.

## Setup

This notebook uses Pyomo with the `ipopt` solver. Create the environment from `pyomo_env.yml` before running.

In [1]:
from pyomo.environ import *

## Model 1: a single price for all seven days

The same price $p$ applies every day. Each day's attendance is bounded by capacity $C$ and by linear demand $D_i + m_i p$. We maximize total revenue $p \sum_i x_i$.

In [2]:
# Capacity, demand intercepts, and price slopes (m_i < 0)
C = 1000
D = [3100, 1500, 1400, 1510, 2000, 2500, 3300]
m = [-62, -50, -40, -42, -52.6, -55.6, -60]

model = ConcreteModel()
model.p = Var(domain=NonNegativeReals)
model.x = Var(range(7), domain=NonNegativeReals)

model.obj = Objective(expr=model.p * sum(model.x[i] for i in range(7)), sense=maximize)

model.constraints = ConstraintList()
for i in range(7):
    model.constraints.add(expr=model.x[i] <= C)
    model.constraints.add(expr=model.x[i] <= D[i] + m[i] * model.p)

SolverFactory('ipopt').solve(model)

print("Optimized p:", round(model.p()))
print("Optimized x values:", [round(model.x[i]()) for i in range(7)])
print("Maximal value of the objective:", round(model.obj()))

Optimized p: 25
Optimized x values: [1000, 226, 380, 440, 659, 1000, 1000]
Maximal value of the objective: 119919


## Model 2: a separate price for each day

Now each day gets its own price $p_i$, so the objective becomes $\sum_i p_i x_i$. Demand on each day still depends only on that day's own price. Compare the total revenue here to Model 1 — and note how little the *average* price actually moves.

In [3]:
model2 = ConcreteModel()
model2.p = Var(range(7), domain=NonNegativeReals)
model2.x = Var(range(7), domain=NonNegativeReals)

model2.obj = Objective(expr=sum(model2.p[i] * model2.x[i] for i in range(7)), sense=maximize)

model2.constraints = ConstraintList()
for i in range(7):
    model2.constraints.add(expr=model2.x[i] <= C)
    model2.constraints.add(expr=model2.x[i] <= D[i] + m[i] * model2.p[i])

SolverFactory('ipopt').solve(model2)

for i in range(7):
    print(f"Day {i+1}:")
    print("Optimized p:", round(model2.p[i]()))
    print("Optimized x value:", round(model2.x[i]()))
print("Maximal value of the objective:", round(model2.obj()))

Day 1:
Optimized p: 34
Optimized x value: 1000
Day 2:
Optimized p: 15
Optimized x value: 750
Day 3:
Optimized p: 18
Optimized x value: 700
Day 4:
Optimized p: 18
Optimized x value: 755
Day 5:
Optimized p: 19
Optimized x value: 1000
Day 6:
Optimized p: 27
Optimized x value: 1000
Day 7:
Optimized p: 38
Optimized x value: 1000
Maximal value of the objective: 155266


## Model 3: day-specific prices with diversion

Phillips calls demand-shifting between periods *diversion*. We add it by letting day $i$'s demand respond to the gap between its price and the others: each day $j$ priced above day $i$ sends a few customers to day $i$, via the term $\sum_j 8\,(p_j - p_i)$. With diversion, customers smooth themselves across the week, so the optimal prices compress toward each other relative to Model 2.

In [24]:
model3 = ConcreteModel()
model3.p = Var(range(7), domain=NonNegativeReals)
model3.x = Var(range(7), domain=NonNegativeReals)

model3.obj = Objective(expr=sum(model3.p[i] * model3.x[i] for i in range(7)), sense=maximize)

model3.constraints = ConstraintList()
for i in range(7):
    model3.constraints.add(expr=model3.x[i] <= C)
    # Diversion: demand on day i also responds to its price gap with every other day
    model3.constraints.add(expr=model3.x[i] <= D[i] + m[i] * model3.p[i] + sum(8 * (model3.p[j] - model3.p[i]) for j in range(7)))

SolverFactory('ipopt').solve(model3)

for i in range(7):
    print(f"Day {i+1}:")
    print("Optimized p:", round(model3.p[i]()))
    print("Optimized x value:", round(model3.x[i]()))
print("Maximal value of the objective:", round(model3.obj()))

Day 1:
Optimized p: 29
Optimized x value: 1000
Day 2:
Optimized p: 18
Optimized x value: 889
Day 3:
Optimized p: 19
Optimized x value: 839
Day 4:
Optimized p: 20
Optimized x value: 894
Day 5:
Optimized p: 21
Optimized x value: 1000
Day 6:
Optimized p: 25
Optimized x value: 1000
Day 7:
Optimized p: 31
Optimized x value: 1000
Maximal value of the objective: 156485
